# 05 · Config 3: agente com ferramentas

Fluxo: **roteador** (LLM, JSON com `ferramenta` e argumentos) → **ferramenta** → **redação** (LLM, só com o resultado da ferramenta).

- `buscar_base`: mesmo fluxo do RAG (barreira 1: pré-filtro por score; barreira 2: o LLM recusa).
- `consultar_gtfs`: `GTFS("base/gtfs").linhas_diretas(origem, destino)`, determinística. Pós-checagem: linha citada que não está no resultado → `linha_inventada`.
- `previsao_onibus`: Olho Vivo; sem `SPTRANS_TOKEN` devolve "indisponível" e o agente recusa ("Não tenho previsão em tempo real agora.").
- `nenhuma`: recusa.

O `gtfs.py` **não foi corrigido** de propósito: os falsos positivos dele aparecem na avaliação.
Resultados em `avaliacao/resultados/agente.jsonl` (incremental, retoma se interrompido).

In [1]:
import json, sys
from collections import Counter
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src')); sys.path.insert(0, str(RAIZ / 'avaliacao'))

import avaliar as av
from sonar_ia import agente as ag
from sonar_ia.gtfs import _tokens
from sonar_ia.llm import nome_modelo

casos = av.carregar_casos()
por_id = {c['id']: c for c in casos}
print('modelo LLM:', nome_modelo(), '| casos:', len(casos), '| SPTRANS_TOKEN definido:', bool(__import__('os').environ.get('SPTRANS_TOKEN')))

modelo LLM: qwen2.5:3b | casos: 63 | SPTRANS_TOKEN definido: False


In [2]:
ag.gtfs()
reg = av.avaliar('agente', ag.agente, casos, verbose=False)
rag = [json.loads(l) for l in (RAIZ / 'avaliacao' / 'resultados' / 'rag.jsonl').read_text(encoding='utf-8').splitlines()]
print(len(reg), 'casos do agente;', len(rag), 'casos do rag (config 2, para comparar)')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

63 casos do agente; 63 casos do rag (config 2, para comparar)


## 1. Acurácia do roteador

Esperado: `texto` → `buscar_base`; `rota` → `consultar_gtfs`; `nao_consta` → qualquer ferramenta, desde que o agente **recuse**.

In [3]:
def roteou_certo(r):
    f, t = r['saida']['ferramenta'], r['tipo']
    if t == 'texto': return f == 'buscar_base'
    if t == 'rota': return f == 'consultar_gtfs'
    return r['saida']['recusou']

ok = {t: [roteou_certo(r) for r in reg if r['tipo'] == t] for t in ('texto', 'rota', 'nao_consta')}
for t, v in ok.items():
    print(f'{t:<11} {sum(v)}/{len(v)} = {sum(v)/len(v):.3f}')
print(f'geral       {sum(map(sum, ok.values()))}/{len(reg)} = {sum(map(sum, ok.values()))/len(reg):.3f}')
print('JSON do roteador inválido:', sum(not r['saida']['roteador_valido'] for r in reg))
print('\nFerramenta escolhida por tipo de caso:')
print(pd.crosstab(pd.Series([r['tipo'] for r in reg], name='tipo'), pd.Series([r['saida']['ferramenta'] for r in reg], name='ferramenta')).to_string())
print('\nRoteamento errado:')
for r in reg:
    if not roteou_certo(r):
        print(f"  {r['id']:<14} -> {r['saida']['ferramenta']:<16} recusou={r['saida']['recusou']!s:<5} {r['pergunta']}")

texto       27/30 = 0.900
rota        16/18 = 0.889
nao_consta  14/15 = 0.933
geral       57/63 = 0.905
JSON do roteador inválido: 0

Ferramenta escolhida por tipo de caso:
ferramenta  buscar_base  consultar_gtfs  nenhuma  previsao_onibus
tipo                                                             
nao_consta            9               0        5                1
rota                  0              16        2                0
texto                27               0        3                0

Roteamento errado:
  texto_23       -> nenhuma          recusou=True  quais documentos eu preciso levar pra me cadastrar no atende+?
  texto_25       -> nenhuma          recusou=True  quantos por cento dos taxis tem que ser acessíveis?
  texto_29       -> nenhuma          recusou=True  quantas estações da cptm são acessiveis?
  rota_06        -> nenhuma          recusou=True  tem trem direto da lapa pro terminal pirituba?
  rota_14        -> nenhuma          recusou=True  tem trem direto da 

## 2. Agente × RAG por tipo (e a config 1 oficial como referência)

In [4]:
sem = [json.loads(l) for l in (RAIZ / 'avaliacao' / 'resultados' / 'sem_contexto.jsonl').read_text(encoding='utf-8').splitlines()]
cfgs = {'1 sem_contexto': sem, '2 rag': rag, '3 agente': reg}
cols = ['acuracia', 'alucinacao', 'recusa_correta', 'falsa_recusa', 'json_valido', 'citacao_correta', 'latencia_media_s']
print('GERAL'); print(pd.DataFrame({k: av.resumo(v) for k, v in cfgs.items()}).T[cols].to_string())
for t in ('texto', 'rota', 'nao_consta'):
    print(f'\n{t.upper()}')
    print(pd.DataFrame({k: av.resumo([r for r in v if r['tipo'] == t]) for k, v in cfgs.items()}).T[cols].to_string())
print('\nCasos em que agente e rag diferem no acerto:')
rag_por_id = {r['id']: r for r in rag}
for r in reg:
    a, b = r['metricas']['acerto'], rag_por_id[r['id']]['metricas']['acerto']
    if a != b:
        print(f"  {r['id']:<14} agente={'acerta' if a else 'erra  '}  rag={'acerta' if b else 'erra  '}  {r['pergunta']}")

GERAL
                acuracia  alucinacao  recusa_correta  falsa_recusa  json_valido  citacao_correta  latencia_media_s
1 sem_contexto    0.2381      0.7619          0.0667        0.0000          1.0           0.0000              0.74
2 rag             0.6825      0.0952          0.9333        0.3333          1.0           0.8000              2.87
3 agente          0.7460      0.1587          0.9333        0.1429          1.0           0.6667              1.87

TEXTO
                acuracia  alucinacao  recusa_correta  falsa_recusa  json_valido  citacao_correta  latencia_media_s
1 sem_contexto    0.3667      0.6333             NaN        0.0000          1.0           0.0000              0.76
2 rag             0.7667      0.1667             NaN        0.0667          1.0           0.8000              4.50
3 agente          0.6667      0.1667             NaN        0.1667          1.0           0.6667              2.03

ROTA
                acuracia  alucinacao  recusa_correta  falsa_r

## 3. Casos de rota: onde estão os erros

Para cada rota que não acertou, a primeira causa que se aplica, nesta ordem:

1. **ROTEAMENTO**: o roteador não escolheu `consultar_gtfs`.
2. **EXTRAÇÃO**: escolheu `consultar_gtfs`, mas origem/destino extraídos não equivalem aos esperados (mesmas palavras-chave do local, ou a ferramenta devolve o mesmo resultado com os dois pares).
3. **FERRAMENTA**: roteou e extraiu certo, mas a resposta errou. Subcausas: *falso positivo do GTFS* (a resposta cita linha de `linhas_incorretas`), *linha inventada pelo LLM* (cita linha que não veio da ferramenta), *recusa indevida* ou *não citou nenhuma linha esperada*.

In [5]:
gt = ag.gtfs()
def equivalente(extraido, esperado):
    return bool(_tokens(extraido)) and _tokens(extraido) == _tokens(esperado)

def mesmo_resultado(o1, d1, o2, d2):
    a, b = gt.linhas_diretas(o1, d1), gt.linhas_diretas(o2, d2)
    return a.get('encontrado') == b.get('encontrado') and [l['linha'] for l in a['linhas']] == [l['linha'] for l in b['linhas']]

categorias = {}
for r in reg:
    if r['tipo'] != 'rota' or r['metricas']['acerto']:
        continue
    c, s = por_id[r['id']], r['saida']
    if s['ferramenta'] != 'consultar_gtfs':
        categorias[r['id']] = ('ROTEAMENTO', f"escolheu {s['ferramenta']}")
        continue
    o, d = s['argumentos'].get('origem', ''), s['argumentos'].get('destino', '')
    if not (equivalente(o, c['origem']) and equivalente(d, c['destino'])) and not (o and d and mesmo_resultado(o, d, c['origem'], c['destino'])):
        categorias[r['id']] = ('EXTRAÇÃO', f"extraiu {o!r} -> {d!r}; esperado {c['origem']!r} -> {c['destino']!r}")
        continue
    texto = s['resposta']
    citadas_incorretas = [l for l in c['linhas_incorretas'] if av.cita_linha(texto, l)]
    if citadas_incorretas:
        categorias[r['id']] = ('FERRAMENTA', f"falso positivo do GTFS citado: {citadas_incorretas}")
    elif s['linha_inventada']:
        categorias[r['id']] = ('FERRAMENTA', f"linha inventada pelo LLM: {s.get('linhas_inventadas')}")
    elif s['recusou']:
        categorias[r['id']] = ('FERRAMENTA', 'recusa indevida (a ferramenta não devolveu linha esperada)')
    else:
        categorias[r['id']] = ('FERRAMENTA', 'não citou nenhuma linha esperada')

rotas = [r for r in reg if r['tipo'] == 'rota']
print(f"rotas: {len(rotas)} | acertos: {sum(r['metricas']['acerto'] for r in rotas)} | erros: {len(categorias)}")
print(Counter(v[0] for v in categorias.values()).most_common())
print(Counter(v[1].split(':')[0] for v in categorias.values() if v[0] == 'FERRAMENTA').most_common())
print()
for id_, (cat, motivo) in categorias.items():
    print(f"{id_:<8} {cat:<11} {motivo}")

rotas: 18 | acertos: 13 | erros: 5
[('FERRAMENTA', 3), ('ROTEAMENTO', 1), ('EXTRAÇÃO', 1)]
[('falso positivo do GTFS citado', 3)]

rota_04  FERRAMENTA  falso positivo do GTFS citado: ['2582-10', '2769-10']
rota_06  ROTEAMENTO  escolheu nenhuma
rota_08  FERRAMENTA  falso positivo do GTFS citado: ['476A-10']
rota_10  EXTRAÇÃO    extraiu 'Terminal Bandeirantes' -> 'Terminal Princesa Isabel'; esperado 'Terminal Bandeira' -> 'Terminal Princesa Isabel'
rota_12  FERRAMENTA  falso positivo do GTFS citado: ['875C-1', '875C-10']


### Evidência das rotas: linha | route_long_name | embarque → desembarque (o que a ferramenta devolveu e o que o agente respondeu)

In [6]:
for r in rotas:
    c, s = por_id[r['id']], r['saida']
    estado = 'ACERTOU' if r['metricas']['acerto'] else 'ERROU  '
    print(f"\n{r['id']} {estado} {c['origem']} -> {c['destino']}  (esperadas={c['linhas_esperadas']}, incorretas={len(c['linhas_incorretas'])})")
    print('   args:', s['argumentos'], '| ferramenta:', s['ferramenta'])
    sf = s['saida_ferramenta']
    if isinstance(sf, dict) and sf.get('linhas'):
        for l in sf['linhas']:
            marca = 'ESPERADA ' if l['linha'] in c['linhas_esperadas'] else 'INCORRETA' if l['linha'] in c['linhas_incorretas'] else 'neutra   '
            print(f"   {marca} {l['linha']:<9} | {l['nome']:<40} | {l['parada_embarque']} -> {l['parada_desembarque']}")
    elif isinstance(sf, dict):
        print('   ferramenta:', sf.get('motivo', sf))
    print('   resposta:', s['resposta'])


rota_01 ACERTOU Tucuruvi -> Jabaquara  (esperadas=['METRÔ L1'], incorretas=0)
   args: {'origem': 'Tucuruvi', 'destino': 'Jabaquara'} | ferramenta: consultar_gtfs
   ESPERADA  METRÔ L1  | TUCURUVI - JABAQUARA                     | Tucuruvi -> Jabaquara
   resposta: Use a linha L1 do Metrô, sentido Jabaquara, que parte de Tucuruvi e vai até Jabaquara.

rota_02 ACERTOU Vila Madalena -> Paraíso  (esperadas=['METRÔ L2'], incorretas=0)
   args: {'origem': 'Vila Madalena', 'destino': 'Paraiso'} | ferramenta: consultar_gtfs
   ESPERADA  METRÔ L2  | VILA MADALENA - VILA PRUDENTE            | Vila Madalena -> Paraíso
   resposta: Use a linha L2 do Metrô, sentido Vila Prudente, para ir de Vila Madalena até Paraíso.

rota_03 ACERTOU Corinthians-Itaquera -> Palmeiras-Barra Funda  (esperadas=['CPTM L11', 'METRÔ L3'], incorretas=0)
   args: {'origem': 'Corinthians Itaquera', 'destino': 'Palmeiras Barra Funda'} | ferramenta: consultar_gtfs
   ESPERADA  CPTM L11  | ESTUDANTES - BARRA FUNDA           

## 4. Alucinações do agente (todas) e exemplos

In [7]:
alu = [r for r in reg if r['metricas']['alucinacao']]
print(f'alucinações: {len(alu)} de {len(reg)}  | por tipo:', dict(Counter(r['tipo'] for r in alu)), '| por ferramenta:', dict(Counter(r['saida']['ferramenta'] for r in alu)))
print('linha_inventada (pós-checagem):', [r['id'] for r in reg if r['saida']['linha_inventada']])
print('citação inválida:', [r['id'] for r in reg if r['saida']['citacao_invalida']])
for r in alu:
    s = r['saida']
    print(f"\n{r['id']} ({r['tipo']}) ferramenta={s['ferramenta']}  {r['pergunta']}")
    print('   resposta:', s['resposta'], '| fonte:', s['fonte'])

alucinações: 10 de 63  | por tipo: {'texto': 5, 'rota': 4, 'nao_consta': 1} | por ferramenta: {'buscar_base': 6, 'consultar_gtfs': 4}
linha_inventada (pós-checagem): []
citação inválida: ['texto_08', 'nao_consta_01']

texto_03 (texto) ferramenta=buscar_base  como eu entro na estaçao ana rosa pelo terminal urbano, tem elevador?
   resposta: Sim, há elevador para acesso à estação e à plataforma na Ana Rosa (Linha 1-Azul). | fonte: ['metro_elev_l1-azul_ana-rosa']

texto_04 (texto) ferramenta=buscar_base  na estação armenia, pelo terminal urbano sul, é elevador ou rampa?
   resposta: Elevador | fonte: ['metro_elev_l1-azul_armenia']

texto_15 (texto) ferramenta=buscar_base  se o motorista ver uma pessoa cega no ponto, ele é obrigado a parar?
   resposta: Sim, o motorista é obrigado a parar para pessoas com deficiência visual acompanhadas de cão-guia. | fonte: ['lei_caoguia_001']

texto_19 (texto) ferramenta=buscar_base  quando eu for guiado por alguem, em que braço devo segurar?
   resposta

In [8]:
def mostrar(r):
    s = r['saida']
    print(f"{r['id']} ({r['tipo']}) {r['latencia_s']}s  ferramenta={s['ferramenta']}  argumentos={s['argumentos']}  barreira={s['barreira']}")
    print('  pergunta :', r['pergunta'])
    print('  resposta :', s['resposta'])
    print('  fonte    :', s['fonte'], '| recusou:', s['recusou'], '| linha_inventada:', s['linha_inventada'])
    print('  métricas :', {k: v for k, v in r['metricas'].items() if v})

certa = [r for r in reg if r['tipo'] == 'rota' and r['metricas']['acerto'] and r['saida']['ferramenta'] == 'consultar_gtfs'][:1]
fp = [r for r in reg if r['id'] in categorias and categorias[r['id']][0] == 'FERRAMENTA'][:1]
tempo = [r for r in reg if r['saida']['ferramenta'] == 'previsao_onibus'][:1]
for r in certa + fp + tempo:
    mostrar(r); print()

rota_01 (rota) 2.12s  ferramenta=consultar_gtfs  argumentos={'origem': 'Tucuruvi', 'destino': 'Jabaquara'}  barreira=None
  pergunta : qual linha do metrô vai do tucuruvi ate o jabaquara?
  resposta : Use a linha L1 do Metrô, sentido Jabaquara, que parte de Tucuruvi e vai até Jabaquara.
  fonte    : ['gtfs_sptrans'] | recusou: False | linha_inventada: False
  métricas : {'json_valido': True, 'acerto': True}

rota_04 (rota) 3.51s  ferramenta=consultar_gtfs  argumentos={'origem': 'Brás', 'destino': 'Tatuapé'}  barreira=None
  pergunta : o que vai do brás ate o tatuapé?
  resposta : Você pode usar o Trem (CPTM) sentido ESTUDANTES ou o Trem (CPTM) sentido CALMON VIANA. Ou ainda, o Metrô sentido CORINTHIANS - ITAQUERA. Para uma opção mais rápida, use o ônibus 2582-10 sentido Term. Pq. D. Pedro Ii ou o ônibus 2769-10 sentido Metrô Tatuapé.
  fonte    : ['gtfs_sptrans'] | recusou: False | linha_inventada: False
  métricas : {'json_valido': True, 'alucinacao': True}

nao_consta_11 (nao_consta)